In [39]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/predictive-modelling-ds/train_videos.csv
/kaggle/input/competitions/predictive-modelling-ds/sample_submission.csv
/kaggle/input/competitions/predictive-modelling-ds/engagement_daily.csv
/kaggle/input/competitions/predictive-modelling-ds/creators_daily.csv
/kaggle/input/competitions/predictive-modelling-ds/test_videos.csv


In [40]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("*.csv"):
    print(path)

/kaggle/input/competitions/predictive-modelling-ds/train_videos.csv
/kaggle/input/competitions/predictive-modelling-ds/sample_submission.csv
/kaggle/input/competitions/predictive-modelling-ds/engagement_daily.csv
/kaggle/input/competitions/predictive-modelling-ds/creators_daily.csv
/kaggle/input/competitions/predictive-modelling-ds/test_videos.csv


**1. Import libraries**

In [41]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.linear_model import HuberRegressor

**2. Load the data**

In [42]:
data_dir = Path("/kaggle/input/competitions/predictive-modelling-ds")

required = [
    "train_videos.csv",
    "test_videos.csv",
    "engagement_daily.csv",
    "sample_submission.csv"
]

missing = [
    name for name in required
    if not (data_dir / name).is_file()
]

if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}"
    )

train = pd.read_csv(
    data_dir / "train_videos.csv",
    dtype={"video_id": "string"}
)

test = pd.read_csv(
    data_dir / "test_videos.csv",
    dtype={"video_id": "string"}
)

engagement = pd.read_csv(
    data_dir / "engagement_daily.csv",
    dtype={"video_id": "string"}
)

sample = pd.read_csv(
    data_dir / "sample_submission.csv",
    dtype={"video_id": "string"}
)

print(
    f"Training: {len(train):,} | "
    f"Test: {len(test):,} | "
    f"Engagement rows: {len(engagement):,}"
)

Training: 12,000 | Test: 3,001 | Engagement rows: 79,489


**3. Extract Day 5 plays**

In [43]:
day5 = (
    engagement.loc[
        engagement["days_since_post"].eq(5),
        ["video_id", "play_count"]
    ]
    .rename(columns={"play_count": "day5_plays"})
)

if day5["video_id"].duplicated().any():
    raise ValueError("Day 5 has duplicate video IDs.")

print(f"Day-5 records: {len(day5):,}")

Day-5 records: 14,751


**4. Merge Day 5 information**

In [44]:
train_features = train.merge(
    day5,
    on="video_id",
    how="left",
    validate="one_to_one"
)

test_features = test.merge(
    day5,
    on="video_id",
    how="left",
    validate="one_to_one"
)

print(
    "Missing Day 5 rows:",
    train_features["day5_plays"].isna().sum(),
    "train;",
    test_features["day5_plays"].isna().sum(),
    "test"
)

Missing Day 5 rows: 210 train; 40 test


**5. Fill missing values**

In [45]:
train_features["day5_plays"] = (
    train_features["day5_plays"].fillna(0)
)

test_features["day5_plays"] = (
    test_features["day5_plays"].fillna(0)
)

print("Missing values filled with 0.")

Missing values filled with 0.


**6. Select features and create Huber model**

In [46]:
features = [
    "duration",
    "day5_plays"
]

model = HuberRegressor(
    epsilon=1.35,
    max_iter=2000
)

print("Features:", features)
print("Model: Huber Regression")

Features: ['duration', 'day5_plays']
Model: Huber Regression


**7. Train the model**

In [47]:
model.fit(
    train_features[features],
    train_features["target_day30_views"]
)

print("Huber model trained successfully.")

Huber model trained successfully.


**8. Predict test data**

In [48]:
predictions = model.predict(
    test_features[features]
)

predictions = np.clip(
    predictions,
    0,
    None
)

print(f"Created {len(predictions):,} predictions.")

Created 3,001 predictions.


**9. Check predictions**

In [49]:
print(
    pd.Series(predictions).describe())

if not np.isfinite(predictions).all():
    raise ValueError("Predictions contain invalid values.")

print("Prediction check passed.")

count    3.001000e+03
mean     2.221784e+04
std      1.694700e+05
min      0.000000e+00
25%      3.425938e+02
50%      6.485086e+02
75%      2.114359e+03
max      5.547065e+06
dtype: float64
Prediction check passed.


**10. Create submission file**

In [50]:
prediction_map = pd.Series(
    predictions,
    index=test_features["video_id"]
)

submission = sample.copy()

submission["target_day30_views"] = (
    submission["video_id"].map(prediction_map)
)

if set(submission["video_id"]) != set(test["video_id"]):
    raise ValueError(
        "Sample submission IDs do not match test IDs."
    )

if submission["target_day30_views"].isna().any():
    raise ValueError(
        "A test video is missing its prediction."
    )

if not np.isfinite(
    submission["target_day30_views"].to_numpy()
).all():
    raise ValueError(
        "Submission contains an invalid prediction."
    )

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(
    f"Saved submission.csv: "
    f"{len(submission):,} rows"
)

display(submission.head())

Saved submission.csv: 3,001 rows


,video_id,target_day30_views
0,7400582591771938090,227.667436
1,7403167206978374958,384.040253
2,7435124823820356906,2334.379337
3,7409800970143714606,300.620454
4,7410935177553333550,275.564123
